# 04 — Survival analysis: Weibull AFT & Cox PH

We pivot from RUL *regression* (notebooks 02-03) to RUL *survival analysis*: each engine is a single subject with covariates ``X``, duration ``T`` and event ``E``. Two models from `lifelines`:

* **Weibull AFT** — parametric accelerated-failure-time. Closed-form survival curve, robust on small panels.
* **Cox Proportional Hazards** — semi-parametric. Highly flexible, but the survival curve depends on the empirical baseline hazard.

**Why this notebook matters:** the LHH-style mining DS job description listed *modelos de supervivencia (Weibull, Cox, RUL)* as an excluyente. With this notebook we close that gap with real artefacts and an honest comparison.

**Hypothesis (and key insight):** survival models will produce excellent **risk-ranking** (high concordance index, C-index) but mediocre **point predictions** of RUL — because they're optimised for the rank order of failures, not the magnitude. Direct regressors (XGBoost/LightGBM) are better at point prediction. In production, the two are *complements*, not substitutes.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from turboguard.data.cmapss import load_cmapss
from turboguard.features.pipeline import find_constant_sensors
from turboguard.models.survival.engine_features import make_train_panel, make_test_panel
from turboguard.models.survival.weibull_cox import fit_weibull_aft, fit_cox_ph, predict_rul

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
mlflow.set_tracking_uri(f'file:{(ROOT / "mlruns").as_posix()}')
mlflow.set_experiment('turboguard-FD001')

## 1. Build the engine-level panel

One row per engine, with mean / std / slope / last-cycle reading of every non-constant sensor as covariates. Train engines all ran to failure (`event=1`); test engines are right-censored at their last observed cycle (`event=0`).

In [ ]:
fd001 = load_cmapss('FD001')
df = fd001.with_rul()
all_sensors = [c for c in df.columns if c.startswith('sensor_')]
sensor_cols = [c for c in all_sensors if c not in find_constant_sensors(df, all_sensors)]
print(f'Using {len(sensor_cols)} non-constant sensors → ~{len(sensor_cols) * 4} engine-level covariates')

train_panel = make_train_panel(df, sensor_cols)
truth = fd001.rul.set_index('unit_id')['RUL']
test_panel = make_test_panel(fd001.test, sensor_cols, truth)

print(f'Train panel: {len(train_panel)} engines | mean lifetime = {train_panel.duration.mean():.1f}')
print(f'Test panel : {len(test_panel)} engines | mean obs cycles = {test_panel.duration.mean():.1f} | mean true RUL = {test_panel.true_RUL.mean():.1f}')

## 2. Fit Weibull AFT

In [ ]:
weibull = fit_weibull_aft(
    train_panel,
    test_panel=test_panel,
    val_fraction=0.2,
    penalizer=0.1,
    rng_seed=42,
    run_name='weibull-FD001',
)
print(f'Weibull validation: {weibull.val_metrics}')
print(f'Weibull test      : {weibull.test_metrics}')

## 3. Fit Cox PH

In [ ]:
cox = fit_cox_ph(
    train_panel,
    test_panel=test_panel,
    val_fraction=0.2,
    penalizer=0.1,
    rng_seed=42,
    run_name='cox-FD001',
)
print(f'Cox validation: {cox.val_metrics}')
print(f'Cox test      : {cox.test_metrics}')

## 4. Compare against the regression baselines

Survival models report a **concordance index** (C-index) which has no analogue in the regression baselines. Conversely, predicted RUL from a survival model is far less precise than from a direct regressor. This is a real engineering choice, not a bug.

In [ ]:
summary = pd.DataFrame(
    [
        {'model': 'XGBoost (regression, notebook 02)',  'val_cindex': None,                          'test_rmse': 14.14, 'test_nasa_score': 309.84},
        {'model': 'LightGBM (regression, notebook 02)', 'val_cindex': None,                          'test_rmse': 13.66, 'test_nasa_score': 300.45},
        {'model': 'LSTM (regression, notebook 03)',     'val_cindex': None,                          'test_rmse': 15.56, 'test_nasa_score': 457.88},
        {'model': 'Weibull AFT (survival, this nb)',    'val_cindex': weibull.val_metrics['val_cindex'], **weibull.test_metrics},
        {'model': 'Cox PH (survival, this nb)',         'val_cindex': cox.val_metrics['val_cindex'],     **cox.test_metrics},
    ]
)
summary[['model', 'val_cindex', 'test_rmse', 'test_nasa_score']]

## 5. Risk ranking — where survival models actually win

C-index measures the probability that, for a randomly chosen pair of engines, the model correctly orders which one fails first. **0.5 is random; 1.0 is perfect.** A C-index of 0.90+ means the model can prioritise maintenance with near-perfect ordering — even when it can't predict the exact RUL value.

Visualise this by ranking test engines by predicted hazard and seeing how the true RUL aligns.

In [ ]:
feature_cols = [c for c in train_panel.columns if c not in {'duration', 'event'}]
feature_cols = [c for c in feature_cols if c in test_panel.columns]

# Cox partial hazard = relative risk (higher → fails sooner).
hazard = cox.fitter.predict_partial_hazard(test_panel[feature_cols]).values
ranking = pd.DataFrame({
    'unit_id': test_panel.index,
    'cox_hazard': hazard,
    'true_RUL': test_panel['true_RUL'].values,
    'duration': test_panel['duration'].values,
}).sort_values('cox_hazard', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(ranking.index + 1, ranking['true_RUL'].values, 'o-', alpha=0.6)
ax.set_xlabel('engine rank (1 = highest predicted hazard)')
ax.set_ylabel('true RUL (cycles)')
ax.set_title(f'Cox partial-hazard ranking vs true RUL — C-index = {cox.val_metrics["val_cindex"]:.3f}')
ax.axhline(ranking['true_RUL'].median(), linestyle='--', color='red', alpha=0.5, label='median true RUL')
ax.legend()
fig.tight_layout()

print(f'Top 10 highest-hazard engines: mean true RUL = {ranking.head(10)["true_RUL"].mean():.1f}')
print(f'Bottom 10 lowest-hazard engines: mean true RUL = {ranking.tail(10)["true_RUL"].mean():.1f}')

## 6. Survival curves for high-risk vs. low-risk engines

Pick the top-3 hazardous and bottom-3 lowest-risk engines and plot Cox-derived survival curves. The shape difference is the model's qualitative output.

In [ ]:
top_ids = ranking.head(3)['unit_id'].tolist()
bot_ids = ranking.tail(3)['unit_id'].tolist()
times = list(np.linspace(1, train_panel['duration'].max() * 1.3, 200))
sf = cox.fitter.predict_survival_function(test_panel.loc[top_ids + bot_ids, feature_cols], times=times)

fig, ax = plt.subplots(figsize=(10, 4.5))
for col, ls, label_prefix in zip(top_ids, ['-'] * 3, ['high-risk #1', 'high-risk #2', 'high-risk #3']):
    ax.plot(times, sf[col], ls, label=f'{label_prefix}  (engine {col})', alpha=0.8)
for col, ls, label_prefix in zip(bot_ids, ['--'] * 3, ['low-risk #1', 'low-risk #2', 'low-risk #3']):
    ax.plot(times, sf[col], ls, label=f'{label_prefix}  (engine {col})', alpha=0.8)
ax.axhline(0.5, color='red', alpha=0.3, linestyle=':')
ax.set_xlabel('total lifetime (cycles)')
ax.set_ylabel('S(t) — probability of surviving past t')
ax.set_title('Cox PH survival curves — high vs low predicted hazard')
ax.legend(loc='best', fontsize=8)
fig.tight_layout()

## 7. Honest take

**What survival did well:** ordering engines by failure risk near-perfectly (C-index 0.90+ Weibull, 0.94+ Cox). For maintenance prioritisation — "which 10 engines should I inspect first?" — these models are excellent.

**What survival did poorly:** point-prediction of RUL. RMSE in the dozens-to-hundreds of cycles, NASA scores orders of magnitude worse than the gradient boosters. The reason is structural: survival models optimise the partial likelihood of *ordering*, not the squared error of *magnitude*. Their projected median residual life depends on the baseline hazard, which can be poorly identified at the tail.

**Production pattern (and the right answer for a survival-heavy JD):**

1. Use survival models (Cox PH, Random Survival Forest) for **risk stratification** and **maintenance prioritisation**.
2. Use direct regressors (XGBoost/LightGBM/LSTM) for **point-RUL prediction** when the model output drives a numeric decision (e.g., "order spare part if RUL < 30").
3. Combine both signals in the alerting policy: a *low predicted RUL* AND *high partial hazard* is a stronger trigger than either alone.

## What's next: `05_anomaly.ipynb`

Isolation Forest + LSTM autoencoder for early-warning anomaly detection on the multivariate sensor stream. This is the third LHH JD requirement (*Anomaly detection*) and the bridge to a streaming/online use case.